In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/post_cell_6.pickle

trying: ['load_nation']
me:  7
trying: ['nation']
me:  7
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['load_lineitem']
me:  2
trying: ['lineitem']


me:  2
trying: ['load_region']
me:  9
trying: ['DATA_ROOT']
me:  0
trying: ['region']
me:  9
trying: ['supplier']
me:  3
trying: ['load_supplier']
me:  3
trying: ['load_part']
me:  1
trying: ['part']
me:  1
trying: ['tpch_parent']
me:  0
trying: ['orders']


me:  4
trying: ['load_orders']
me:  4
trying: ['orig_output']
me:  6
trying: ['customer']
me:  5
trying: ['load_customer']
me:  5


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_part
Declaring variable part
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable supplier
Declaring variable load_supplier
Declaring variable orders
Declaring variable load_orders
Declaring variable customer
Declaring variable load_customer
Declaring variable orig_output
Declaring variable load_nation
Declaring variable nation
Declaring variable load_region
Declaring variable region


In [4]:
%%RecordEvent
%%time
### cell 7 ###

# Optimized for cudf: eliminate UDF and CPU apply by computing aggregates on GPU
# Filter PART and select key
part_filtered = part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]

# Prepare LINEITEM with volume
lineitem_filtered = lineitem[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
lineitem_filtered["VOLUME"] = lineitem_filtered.L_EXTENDEDPRICE * (1.0 - lineitem_filtered.L_DISCOUNT)
lineitem_filtered = lineitem_filtered[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Join PART and LINEITEM
total = part_filtered.merge(
    lineitem_filtered,
    left_on="P_PARTKEY",
    right_on="L_PARTKEY",
    how="inner"
)[["L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Join SUPPLIER to get supplier nation
total = total.merge(
    supplier[["S_SUPPKEY", "S_NATIONKEY"]],
    left_on="L_SUPPKEY",
    right_on="S_SUPPKEY",
    how="inner"
)[["L_ORDERKEY", "VOLUME", "S_NATIONKEY"]]

# Filter ORDERS by date and extract year
orders_filtered = orders[
    (orders.O_ORDERDATE >= "1995-01-01") & (orders.O_ORDERDATE < "1997-01-01")
]
orders_filtered["O_YEAR"] = orders_filtered.O_ORDERDATE.dt.year
orders_filtered = orders_filtered[["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]

# Join ORDERS
total = total.merge(
    orders_filtered,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_CUSTKEY", "O_YEAR"]]

# Join CUSTOMER to get customer nation
total = total.merge(
    customer[["C_CUSTKEY", "C_NATIONKEY"]],
    left_on="O_CUSTKEY",
    right_on="C_CUSTKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_YEAR", "C_NATIONKEY"]]

# Join NATION for customer region
total = total.merge(
    nation[["N_NATIONKEY", "N_REGIONKEY"]],
    left_on="C_NATIONKEY",
    right_on="N_NATIONKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_YEAR", "N_REGIONKEY"]]

# Join NATION to get supplier nation name
n2 = nation[["N_NATIONKEY", "N_NAME"]].rename(columns={"N_NAME": "NATION"})
total = total.merge(
    n2,
    left_on="S_NATIONKEY",
    right_on="N_NATIONKEY",
    how="inner"
)[["VOLUME", "O_YEAR", "N_REGIONKEY", "NATION"]]

# Filter REGION = AMERICA
region_filtered = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]
total = total.merge(
    region_filtered,
    left_on="N_REGIONKEY",
    right_on="R_REGIONKEY",
    how="inner"
)[["VOLUME", "O_YEAR", "NATION"]]

# Compute market share: Brazil volume / total volume per year
# Create a column that is volume for Brazil only
total["VOLUME_BRAZIL"] = total.VOLUME * (total.NATION == "BRAZIL")
# Aggregate sums on GPU
agg = total.groupby("O_YEAR").agg({
    "VOLUME": "sum",
    "VOLUME_BRAZIL": "sum"
})
# Compute share and prepare final result
agg["MKT_SHARE"] = agg.VOLUME_BRAZIL / agg.VOLUME
total = agg["MKT_SHARE"].reset_index().sort_values("O_YEAR")

CPU times: user 95.7 ms, sys: 56.1 ms, total: 152 ms
Wall time: 164 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_0.pickle

migration speed (bps): 805860146.0013206
---------------------------
variables to migrate:
total 48
DATA_ROOT 80
STORAGE_OPTS 64
region 48
load_region 144
n2 48
pd 72
agg 48
orders_filtered 48
load_supplier 144
region_filtered 48
supplier 48
lineitem_filtered 48
part_filtered 48
load_lineitem 144
lineitem 48
orders 48
load_part 144
load_orders 144
part 48
load_customer 144
customer 48
tpch_parent 54
orig_output 16
load_nation 144
nation 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high/checkpoints/post_cell_7_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem', 'part']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem', 'part', 'supplier']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables ['lineitem', 'part', 'supplier', 'orders']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['lineitem', 'customer', 'supplier', 'part', 'orders']
Modified dataframes
======= Cell 6 =======
Input varia

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q08/opt_cell_exec_info_7_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[7], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/annotated/checkpoints/post_cell_7.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['load_nation']
me:  7
trying: ['nation']
me:  7
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['supplier_filtered']
me:  11
trying: ['region']
me:  9
trying: ['load_region']
me:  9
trying: ['pd']
me:  0
trying: ['n2_filtered']
me:  11
trying: ['load_supplier']
me:  3
trying: ['part_filtered']
me:  11
trying: ['supplier']
me:  3
trying: ['load_lineitem']
me:  2
trying: ['lineitem']


me:  2
trying: ['customer_filtered']
me:  11
trying: ['orders']


me:  4
trying: ['load_part']
me:  1
trying: ['udf']
me:  11
trying: ['load_orders']
me:  4
trying: ['part']
me:  1
trying: ['lineitem_filtered']


me:  11
trying: ['n1_filtered']
me:  11
trying: ['orders_filtered']
me:  11
trying: ['load_customer']
me:  5
trying: ['customer']
me:  5
trying: ['tpch_parent']
me:  0
trying: ['region_filtered']
me:  11
trying: ['total']
me:  11
trying: ['orig_output']
me:  6


Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable pd
Declaring variable tpch_parent
Declaring variable load_part
Declaring variable part
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_supplier
Declaring variable supplier
Declaring variable orders
Declaring variable load_orders
Declaring variable load_customer
Declaring variable customer
Declaring variable orig_output
Declaring variable load_nation
Declaring variable nation
Declaring variable region
Declaring variable load_region
Declaring variable supplier_filtered
Declaring variable n2_filtered
Declaring variable part_filtered
Declaring variable customer_filtered
Declaring variable udf
Declaring variable lineitem_filtered
Declaring variable n1_filtered
Declaring variable orders_filtered
Declaring variable region_filtered
Declaring variable total
